# Batch Publishing

This notebook demonstrates how to use the `DataStoreClient` to batch publish N iterations of data within a given `Step`. Rather than calling `publish_condition` or `publish_measurement` N times to publish data for each of these N iterations, this data can instead be published by a single call to `publish_condition_batch` or `publish_measurement_batch`, respectively. Batch publishing can help improve overall publishing performance.

**Note:** These batching APIs handle batch publishing N iterations of data for a single condition or measurement with the specified name. They do *not* support publishing data across multiple (distinctly named) conditions or multiple (distinctly named) measurements at once.

In [ ]:
# Perform example-specific setup with the DataStoreContext. This is not needed when writing production code.
from utilities import DataStoreContext
data_store_context = DataStoreContext()
data_store_context.initialize()

# Create the TestResult and Step into which we will later batch publish conditions and measurements.
from ni.datastore.data import DataStoreClient, Step, TestResult

data_store_client = DataStoreClient()

test_result = TestResult(name="Batch Publish Example")
test_result_id = data_store_client.create_test_result(test_result)

step = Step(name="Example Step", test_result_id=test_result_id)
step_id = data_store_client.create_step(step)

print(f"Created Test Result: {test_result_id}")
print(f"Created Step: {step_id}")

## Batch Publishing Condition Values

The `publish_condition_batch` method of `DataStoreClient` can be used to publish the value of a given condition across N parametric iterations at once. This usage of `publish_condition_batch` is equivalent to calling `publish_condition` for that same condition N times.

The condition values themselves may be supplied to `publish_condition_batch` as either an `Iterable` or a `Vector`.

Supported element types of the `Iterable` are `float`, `int`, `str`, and `bool`: 

In [ ]:
float_condition_id = data_store_client.publish_condition_batch(
    name="Example Float Condition",
    condition_type="Setup",
    values=[1.25, 2.5, 3.75, 5.0],
    step_id=step_id,
 )

integer_condition_id = data_store_client.publish_condition_batch(
    name="Example Integer Condition",
    condition_type="Setup",
    values=[1, 2, 3, 4],
    step_id=step_id,
 )

string_condition_id = data_store_client.publish_condition_batch(
    name="Example String Condition",
    condition_type="Setup",
    values=["cold", "ambient", "warm", "hot"],
    step_id=step_id,
 )

bool_condition_id = data_store_client.publish_condition_batch(
    name="Example Bool Condition",
    condition_type="Setup",
    values=[True, False, True, False],
    step_id=step_id,
 )

print(f"Published Example Float Condition ID: {float_condition_id}")
print(f"Published Example Integer Condition ID: {integer_condition_id}")
print(f"Published Example String Condition ID: {string_condition_id}")
print(f"Published Example Bool Condition ID: {bool_condition_id}")

Supplying a `Vector` allows the client to specify additional information, such as units:

In [ ]:
from nitypes.vector import Vector

condition_vector = Vector(values=[0.5, 1.0, 1.5, 2.0], units="Amps")

vector_condition_id = data_store_client.publish_condition_batch(
    name="Example Published-As-Vector Condition",
    condition_type="Setup",
    values=condition_vector,
    step_id=step_id,
 )

print(f"Published Example Published-As-Vector Condition ID: {vector_condition_id}")

Published condition values can be read back in the same manner as when reading back condition values that were published individually via `publish_condition`.

More specifically, condition values published via `publish_condition_batch` are read back as a `Vector` containing all N iterations of parametric data published for a particular condition:

In [ ]:
published_float_condition = data_store_client.get_condition(float_condition_id)
published_integer_condition = data_store_client.get_condition(integer_condition_id)
published_string_condition = data_store_client.get_condition(string_condition_id)
published_bool_condition = data_store_client.get_condition(bool_condition_id)
published_as_vector_condition = data_store_client.get_condition(vector_condition_id)

read_back_float_vector = data_store_client.read_condition_value(published_float_condition, expected_type=Vector)
read_back_integer_vector = data_store_client.read_condition_value(published_integer_condition, expected_type=Vector)
read_back_string_vector = data_store_client.read_condition_value(published_string_condition, expected_type=Vector)
read_back_bool_vector = data_store_client.read_condition_value(published_bool_condition, expected_type=Vector)
read_back_vector = data_store_client.read_condition_value(published_as_vector_condition, expected_type=Vector)

print(f"Read Example Float Condition: {read_back_float_vector}")
print(f"Read Example Integer Condition: {read_back_integer_vector}")
print(f"Read Example String Condition: {read_back_string_vector}")
print(f"Read Example Bool Condition: {read_back_bool_vector}")
print(f"Read Example Published-As-Vector Condition: {read_back_vector}")

## Batch Publishing Measurement Values

The `publish_measurement_batch` method of `DataStoreClient` can similarly be used to publish the value of a given measurement across N parametric iterations at once. This usage of `publish_measurement_batch` is equivalent to calling `publish_measurement` for that same measurement N times.

This is conceptually similar to the batch publishing of condition values. One important note, however, is that whereas conditions only support a given publish iteration containing a single scalar value, measurements support both **scalar** and **non-scalar** values.

Examples of supported non-scalar types are `AnalogWaveform` and `Vector`. The process of batch publishing both scalar and non-scalar measurement values is similar, though as is the case for publishing scalar and non-scalar measurement values using the non-batched `publish_measurement` method, the process of later reading that measurement data from Measurement Data Services is slightly different between the two cases, as shown below.

### Scalar Measurements

Scalar measurement values may be supplied to `publish_measurement_batch` as either an `Iterable` or a `Vector`.

Supported element types of the `Iterable` are `float`, `int`, `str`, and `bool`: 

In [ ]:
float_measurement_ids = data_store_client.publish_measurement_batch(
    name="Example Float Measurement",
    values=[0.125, 0.25, 0.5, 1.0],
    step_id=step_id,
 )

integer_measurement_ids = data_store_client.publish_measurement_batch(
    name="Example Integer Measurement",
    values=[10, 20, 30, 40],
    step_id=step_id,
 )

string_measurement_ids = data_store_client.publish_measurement_batch(
    name="Example String Measurement",
    values=["nominal", "warning", "critical", "retest"],
    step_id=step_id,
 )

bool_measurement_ids = data_store_client.publish_measurement_batch(
    name="Example Bool Measurement",
    values=[False, False, True, True],
    step_id=step_id,
 )

print(f"Published Example Float Measurement IDs: {float_measurement_ids}")
print(f"Published Example Integer Measurement IDs: {integer_measurement_ids}")
print(f"Published Example String Measurement IDs: {string_measurement_ids}")
print(f"Published Example Bool Measurement IDs: {bool_measurement_ids}")

Note that because these are scalar measurements, only a single measurement ID is present in the returned `Sequence` from each publish call. For non-scalar measurements, the `publish_measurement_batch` method returns a `Sequence` of N IDs, as we will see further below.

Supplying a `Vector` allows the client to specify additional information, such as units:

In [ ]:
measurement_vector = Vector(values=[1.2, 1.4, 1.6, 1.8], units="Volts")

vector_measurement_ids = data_store_client.publish_measurement_batch(
    name="Example Published-As-Vector Measurement",
    values=measurement_vector,
    step_id=step_id,
 )

print(f"Published Example Published-As-Vector Measurement IDs: {vector_measurement_ids}")

Published measurement values can be read back in the same manner as when reading back measurement values that were published individually via `publish_measurement`.

More specifically, scalar measurement values published via `publish_measurement_batch` are read back as a `Vector` containing all N iterations of parametric data published for a particular measurement:

In [ ]:
published_float_measurement = data_store_client.get_measurement(float_measurement_ids[0])
published_integer_measurement = data_store_client.get_measurement(integer_measurement_ids[0])
published_string_measurement = data_store_client.get_measurement(string_measurement_ids[0])
published_bool_measurement = data_store_client.get_measurement(bool_measurement_ids[0])
published_as_vector_measurement = data_store_client.get_measurement(vector_measurement_ids[0])

read_back_float_measurement = data_store_client.read_measurement_value(published_float_measurement, expected_type=Vector)
read_back_integer_measurement = data_store_client.read_measurement_value(published_integer_measurement, expected_type=Vector)
read_back_string_measurement = data_store_client.read_measurement_value(published_string_measurement, expected_type=Vector)
read_back_bool_measurement = data_store_client.read_measurement_value(published_bool_measurement, expected_type=Vector)
read_back_vector_measurement = data_store_client.read_measurement_value(published_as_vector_measurement, expected_type=Vector)

print(f"Read Float Measurement: {read_back_float_measurement}")
print(f"Read Integer Measurement: {read_back_integer_measurement}")
print(f"Read String Measurement: {read_back_string_measurement}")
print(f"Read Bool Measurement: {read_back_bool_measurement}")
print(f"Read Published-As-Vector Measurement: {read_back_vector_measurement}")

### Non-Scalar Measurements

Unlike condition values, a measurement value for a particular publish iteration may be non-scalar, such as an `AnalogWaveform` or `Vector`.

To batch publish non-scalar values for a particular measurement, supply `publish_measurement_batch` with an `Iterable` containing N non-scalar elements. Each of the N elements in the `Iterable` will correspond to a single publish iteration. This is equivalent to calling the non-batched `publish_measurement` method N times for the measurement in question:

In [ ]:
from datetime import timezone
import hightime as ht
import numpy as np
from nitypes.waveform import AnalogWaveform, Timing

# Batch publish two AnalogWaveform (i.e., non-scalar) values for the Example AnalogWaveform Measurement.
waveforms = [
    AnalogWaveform(
        sample_count=4,
        raw_data=np.array([0.0, 0.25, 0.5, 0.75], dtype=np.float64),
        timing=Timing.create_with_regular_interval(
            ht.timedelta(seconds=1e-3),
            ht.datetime.now(timezone.utc),
        ),
    ),
    AnalogWaveform(
        sample_count=4,
        raw_data=np.array([1.0, 0.85, 0.65, 0.4], dtype=np.float64),
        timing=Timing.create_with_regular_interval(
            ht.timedelta(seconds=1e-3),
            ht.datetime.now(timezone.utc),
        ),
    ),
]

waveform_measurement_ids = data_store_client.publish_measurement_batch(
    name="Example AnalogWaveform Measurement",
    values=waveforms,
    step_id=step_id,
 )

# Batch publish two Vector (i.e., non-scalar) values for the Example Vector Measurement.
vector_measurements = [
    Vector(values=[1.0, 1.25, 1.5], units="Volts"),
    Vector(values=[2.0, 2.25, 2.5], units="Volts"),
]

vector_measurement_ids = data_store_client.publish_measurement_batch(
    name="Example Vector Measurement",
    values=vector_measurements,
    step_id=step_id,
 )

print(f"Published Example AnalogWaveform Measurement IDs: {waveform_measurement_ids}")
print(f"Published Example Vector Measurement IDs: {vector_measurement_ids}")

Published non-scalar measurement values can be read back in the same manner as when reading back measurement values that were published individually via `publish_measurement`.

More specifically, non-scalar measurement values published via `publish_measurement_batch` are read back **individually**, with each published value corresponding to a separate `PublishedMeasurement`.

The `parametric_index` of each `PublishedMeasurement` indicates the publish iteration to which it corresponds:

In [ ]:
published_waveform_measurement_0 = data_store_client.get_measurement(waveform_measurement_ids[0])
published_waveform_measurement_1 = data_store_client.get_measurement(waveform_measurement_ids[1])
published_vector_measurement_0 = data_store_client.get_measurement(vector_measurement_ids[0])
published_vector_measurement_1 = data_store_client.get_measurement(vector_measurement_ids[1])

read_back_waveform_measurement_0 = data_store_client.read_measurement_value(published_waveform_measurement_0, expected_type=AnalogWaveform)
read_back_waveform_measurement_1 = data_store_client.read_measurement_value(published_waveform_measurement_1, expected_type=AnalogWaveform)
read_back_vector_measurement_0 = data_store_client.read_measurement_value(published_vector_measurement_0, expected_type=Vector)
read_back_vector_measurement_1 = data_store_client.read_measurement_value(published_vector_measurement_1, expected_type=Vector)

# The 'parametric_index' field of the PublishedMeasurement indicates
# which publish iteration the non-scalar measurement value corresponds to.
# In this example, we published two values for each (uniquely named) measurement,
# so the PublishedMeasurement corresponding to the first publish for each measurement 
# has a 'parametric_index' of 0 and the PublishedMeasurement corresponding to the
# second publish for each measurement has a 'parametric_index' of 1.
print(
    f"{published_waveform_measurement_0.name} at Parametric Index "
    f"{published_waveform_measurement_0.parametric_index}: "
    f"{read_back_waveform_measurement_0.raw_data.tolist()}"
 )
print(
    f"{published_waveform_measurement_1.name} at Parametric Index "
    f"{published_waveform_measurement_1.parametric_index}: "
    f"{read_back_waveform_measurement_1.raw_data.tolist()}"
 )

print(
    f"{published_vector_measurement_0.name} at Parametric Index "
    f"{published_vector_measurement_0.parametric_index}: "
    f"{read_back_vector_measurement_0}"
 )
print(
    f"{published_vector_measurement_1.name} at Parametric Index "
    f"{published_vector_measurement_1.parametric_index}: "
    f"{read_back_vector_measurement_1}"
 )

## Clean-Up

Close the `DataStoreClient` and tear down the example-specific context when done.

In [ ]:
data_store_client.close()

# Perform example-specific cleanup. This is not needed when writing production code.
data_store_context.close()